 # The Anchor Dilemma in T20 Cricket

 ## Prerequisites

In [ ]:
%pip install pandas numpy matplotlib scikit-learn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


 ## Stage 1: Data Ingestion

In [ ]:
deliveries_df = pd.read_csv('./dataset/deliveries.csv')
matches_df = pd.read_csv('./dataset/matches.csv')


 ### Intermediate Check: Raw Data Dimensions

In [ ]:
print("Raw Deliveries Shape:", deliveries_df.shape)
print("Raw Matches Shape:", matches_df.shape)


 ## Stage 2: Data Cleaning

 Removing matches with less than 19 completed overs to ensure baseline scoring is consistent, and handling missing values.

In [ ]:
overs_per_match = deliveries_df.groupby(['match_id', 'inning'])['over'].max().reset_index()
full_matches = overs_per_match[overs_per_match['over'] >= 19]['match_id'].unique()

deliveries_df = deliveries_df[deliveries_df['match_id'].isin(full_matches)]
matches_df = matches_df[matches_df['id'].isin(full_matches)]

matches_df = matches_df.dropna(subset=['winner'])


 ### Intermediate Check: Cleaned Data Dimensions

In [ ]:
print("Cleaned Deliveries Shape:", deliveries_df.shape)
print("Cleaned Matches Shape:", matches_df.shape)
print("Total Full Matches Retained:", len(full_matches))


 ## Stage 3: Feature Engineering (The "Anchor" Math)

 ### Part 1: Calculating individual batter statistics per inning

In [ ]:
batter_inning_stats = deliveries_df.groupby(['match_id', 'inning', 'batting_team', 'batter']).agg(
    runs_scored=('batsman_runs', 'sum'),
    balls_faced=('ball', 'count')
).reset_index()

batter_inning_stats['strike_rate'] = (batter_inning_stats['runs_scored'] / batter_inning_stats['balls_faced']) * 100
batter_inning_stats['is_anchor'] = (batter_inning_stats['balls_faced'] > 30) & (batter_inning_stats['strike_rate'] < 125)


 ### Intermediate Analysis: Who are the most frequent "Anchors"?

In [ ]:
anchors_only = batter_inning_stats[batter_inning_stats['is_anchor']]
top_anchor_players = anchors_only['batter'].value_counts().head(5)

print("Top 5 Players with the most 'Anchor' Innings (>30 balls, SR < 125):")
print(top_anchor_players)


 ### Part 2: Separating Anchor vs Non-Anchor performances

In [ ]:
anchor_summary = anchors_only.groupby(['match_id', 'inning'])['balls_faced'].sum().reset_index()
anchor_summary = anchor_summary.rename(columns={'balls_faced': 'anchor_balls_faced'})

non_anchors = batter_inning_stats[~batter_inning_stats['is_anchor']]
non_anchor_summary = non_anchors.groupby(['match_id', 'inning']).agg(
    non_anchor_runs=('runs_scored', 'sum'),
    non_anchor_balls=('balls_faced', 'sum')
).reset_index()

non_anchor_summary['non_anchor_strike_rate'] = (non_anchor_summary['non_anchor_runs'] / non_anchor_summary['non_anchor_balls']) * 100


 ### Part 3: Aggregating team totals and merging features per inning

In [ ]:
deliveries_df['is_wicket'] = deliveries_df['player_dismissed'].notna().astype(int)

inning_totals = deliveries_df.groupby(['match_id', 'inning', 'batting_team']).agg(
    team_final_score=('total_runs', 'sum'),
    wickets_lost=('is_wicket', 'sum')
).reset_index()

final_features = inning_totals.merge(anchor_summary, on=['match_id', 'inning'], how='left')
final_features = final_features.merge(non_anchor_summary[['match_id', 'inning', 'non_anchor_strike_rate']], on=['match_id', 'inning'], how='left')

final_features['anchor_balls_faced'] = final_features['anchor_balls_faced'].fillna(0)
final_features['anchor_present'] = (final_features['anchor_balls_faced'] > 0).astype(int)
final_features['non_anchor_strike_rate'] = final_features['non_anchor_strike_rate'].fillna(0)

win_data = matches_df[['id', 'winner']].rename(columns={'id': 'match_id'})
final_features = final_features.merge(win_data, on='match_id', how='left')
final_features['match_won'] = (final_features['batting_team'] == final_features['winner']).astype(int)


 ### Intermediate Check: Final Features Head

In [ ]:
print(final_features[['match_id', 'team_final_score', 'anchor_present', 'anchor_balls_faced', 'match_won']].head())


 ## Stage 4: Visual Analysis

 ### Plot 1: Anchor Balls vs Team Score

In [ ]:
anchor_innings = final_features[final_features['anchor_present'] == 1]

plt.figure(figsize=(8, 5))
plt.scatter(anchor_innings['anchor_balls_faced'], anchor_innings['team_final_score'], alpha=0.6, color='coral')
plt.title('Anchor Balls Faced vs Team Final Score')
plt.xlabel('Anchor Balls Faced')
plt.ylabel('Team Final Score')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


 ### Plot 2: Spread of Team Scores (Anchor vs No Anchor)

In [ ]:
no_anchor_scores = final_features[final_features['anchor_present'] == 0]['team_final_score']
anchor_scores = final_features[final_features['anchor_present'] == 1]['team_final_score']

plt.figure(figsize=(6, 5))
plt.boxplot([no_anchor_scores, anchor_scores], tick_labels=['No Anchor', 'Anchor Present'], patch_artist=True, boxprops=dict(facecolor='lightblue'))
plt.title('Distribution of Team Scores: Anchor vs No Anchor')
plt.ylabel('Team Final Score')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


 ### Plot 3: Win Percentage Impact

In [ ]:
win_rates = final_features.groupby('anchor_present')['match_won'].mean() * 100

plt.figure(figsize=(6, 4))
plt.bar(['No Anchor', 'Anchor Present'], win_rates, color=['#4CAF50', '#F44336'], width=0.5)
plt.title('Win Percentage: Anchor vs No Anchor')
plt.ylabel('Win Percentage (%)')
plt.ylim(0, 100)
plt.show()


 ## Stage 5: Statistical Modeling

 Building the Multiple Linear Regression model and evaluating its predictive power.

In [ ]:
model_data = final_features.dropna(subset=['team_final_score', 'anchor_balls_faced', 'non_anchor_strike_rate', 'wickets_lost'])

X = model_data[['anchor_balls_faced', 'non_anchor_strike_rate', 'wickets_lost']]
y = model_data['team_final_score']

regression_model = LinearRegression()
regression_model.fit(X, y)

y_pred = regression_model.predict(X)
model_r2 = r2_score(y, y_pred)

anchor_coef = regression_model.coef_[0]
non_anchor_sr_coef = regression_model.coef_[1]
wickets_coef = regression_model.coef_[2]

print("Model R-squared Score:", round(model_r2, 4))
print("Intercept:", round(regression_model.intercept_, 2))
print("Coefficient - Anchor Balls Faced:", round(anchor_coef, 2))
print("Coefficient - Non-Anchor Strike Rate:", round(non_anchor_sr_coef, 2))
print("Coefficient - Wickets Lost:", round(wickets_coef, 2))


 ## Stage 6: Export & Reporting

In [ ]:
final_features.to_csv('anchor_analysis.csv', index=False)

impact_direction = "negatively" if anchor_coef < 0 else "positively"
no_anchor_win_rate = win_rates.get(0, 0)
anchor_win_rate = win_rates.get(1, 0)
most_frequent_anchor = top_anchor_players.index[0]

insights = f"""Final Conclusions:

1. Model Confidence: The regression model explains {model_r2 * 100:.1f}% of the variance in a team's final score.
2. The Run Cost: For every additional ball faced by an 'anchor', the team's expected final score is impacted by {anchor_coef:.2f} runs.
3. Win Probability: Teams playing without an anchor secured a win rate of {no_anchor_win_rate:.0f}%, dropping to {anchor_win_rate:.0f}% when relying on an anchor innings.
4. Player Insight: {most_frequent_anchor} has historically played the most anchor innings under our defined criteria.

Overall, the data proves that playing it safe and consuming balls at a low strike rate {impact_direction} impacts the final team total and win probability in the IPL format.
"""

print(insights)